In [6]:
# %%
# !pip install youtube-transcript-api scrapetube

from youtube_transcript_api import YouTubeTranscriptApi
import scrapetube
from urllib.parse import urlparse, parse_qs
import os

In [ ]:
# %%
# --- Pipeline Configuration ---
# Feed the queue channel handles or URLs instead of hardcoded video links.

experts_queue = {
    # Channels
    "josh_braun": "https://www.youtube.com/@joshbraunsales",
    "jason_bay": "https://www.youtube.com/@jasondbay", 
    "john_barrows": "https://www.youtube.com/@JohnMBarrows",
    "will_allred": "https://www.youtube.com/@itslavenderduh",
    "jed_mahrle": "https://www.youtube.com/@practicalprospecting",
    "becc_holland": "https://www.youtube.com/@flipthescript-sales",
    "eric_nowoslawski": "https://www.youtube.com/@ericnowoslawski",
    "armand_farrokh": "https://www.youtube.com/@30MPC",

    # Guest Spot Video IDs (Direct extraction for high signal)
    "charlotte_johnson": ["Z5mwjeM4YZA", "bq_SWgq9VUk"],
    "florin_tatulea": ["j2z-PISTO9c", "8Fo8nsfNZxQ"]
}

# Number of recent videos to pull per expert
MAX_VIDEOS_PER_EXPERT = 2

OUTPUT_DIR = "../research/youtube-transcripts"

In [8]:
# %%
def extract_video_id(url: str) -> str:
    """Robust URL parser for all YouTube link formats."""
    parsed = urlparse(url)
    if parsed.hostname == 'youtu.be':
        return parsed.path[1:]
    if parsed.hostname in ('www.youtube.com', 'youtube.com'):
        if parsed.path == '/watch':
            return parse_qs(parsed.query).get('v', [None])[0]
    return url

In [9]:
# %%
# Robust Extraction Layer - Checks all content types (Videos, Shorts, Streams)
transcripts_data = []
ytt_api = YouTubeTranscriptApi()

# We'll check these tabs in order of "signal quality"
CONTENT_TYPES = ["videos", "streams", "shorts"]

for expert, channel_url in experts_queue.items():
    if not channel_url: continue
        
    print(f"\nScanning channel for {expert}...")
    found_videos = []
    
    # Try each content type until we hit our MAX_VIDEOS_PER_EXPERT limit
    for c_type in CONTENT_TYPES:
        if len(found_videos) >= MAX_VIDEOS_PER_EXPERT:
            break
            
        try:
            limit_needed = MAX_VIDEOS_PER_EXPERT - len(found_videos)
            vids = scrapetube.get_channel(channel_url=channel_url, 
                                         limit=limit_needed, 
                                         content_type=c_type)
            found_videos.extend(list(vids))
        except Exception as e:
            continue # If a tab doesn't exist, just move to the next

    print(f"  Found {len(found_videos)} recent items across tabs. Extracting...")
    
    for vid in found_videos:
        vid_id = vid['videoId']
        try:
            raw_transcript = ytt_api.fetch(vid_id)
            full_text = " ".join([segment.text for segment in raw_transcript])
            
            transcripts_data.append({
                "expert": expert,
                "video_id": vid_id,
                "url": f"https://www.youtube.com/watch?v={vid_id}",
                "content": full_text.replace(". ", ".\n\n")
            })
            print(f"    ✓ Success: [{vid_id}]")
        except Exception as e:
            print(f"    ✗ Transcript not available for [{vid_id}]: {e}")

print("\nRobust Extraction Complete.")


Scanning channel for josh_braun...
  Found 2 recent items across tabs. Extracting...
    ✗ Transcript not available for [hpXev0vUcvQ]: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=hpXev0vUcvQ! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

Ways to work around this are explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).


If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://gith

In [10]:
# %%%
# Updated Extraction Layer - Handles both Channel URLs and specific Video ID lists
transcripts_data = []
ytt_api = YouTubeTranscriptApi()

CONTENT_TYPES = ["videos", "streams", "shorts"]

for expert, value in experts_queue.items():
    if not value: continue
        
    print(f"\nProcessing {expert}...")
    found_video_ids = []
    
    # Check if we have a list of specific IDs (for Guests like Charlotte/Florin)
    if isinstance(value, list):
        print(f"  Using {len(value)} specific video IDs provided.")
        found_video_ids = value
    
    # Otherwise, scrape the channel URL as usual
    else:
        print(f"  Scanning channel URL: {value}")
        try:
            found_videos = []
            for c_type in CONTENT_TYPES:
                if len(found_videos) >= MAX_VIDEOS_PER_EXPERT: break
                limit_needed = MAX_VIDEOS_PER_EXPERT - len(found_videos)
                vids = scrapetube.get_channel(channel_url=value, limit=limit_needed, content_type=c_type)
                found_videos.extend(list(vids))
            found_video_ids = [v['videoId'] for v in found_videos]
        except Exception as e:
            print(f"    ✗ Error scraping channel: {e}")
            continue

    print(f"  Attempting transcript extraction for {len(found_video_ids)} videos...")
    
    for vid_id in found_video_ids:
        try:
            raw_transcript = ytt_api.fetch(vid_id)
            full_text = " ".join([segment.text for segment in raw_transcript])
            
            transcripts_data.append({
                "expert": expert,
                "video_id": vid_id,
                "url": f"https://www.youtube.com/watch?v={vid_id}",
                "content": full_text.replace(". ", ".\n\n")
            })
            print(f"    ✓ Success: [{vid_id}]")
        except Exception as e:
            print(f"    ✗ Transcript not available for [{vid_id}]: {e}")

print("\nExtraction Complete.")


Processing josh_braun...
  Scanning channel URL: https://www.youtube.com/@joshbraunsales
  Attempting transcript extraction for 2 videos...
    ✗ Transcript not available for [hpXev0vUcvQ]: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=hpXev0vUcvQ! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

Ways to work around this are explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).


If you are sure that the described cause is not responsible for this error and that a transcript should